# Workshop Databricks — Hands-on para Data Engineering e Data Analytics

### Governança, preparação visual, camada semântica, análise e consumo sobre o mesmo conjunto de dados

Este roteiro foi reorganizado para execução **junto com os participantes**. Em vez de apenas explicar cada capability, cada bloco abaixo produz um artefato real no workspace usando o mesmo schema configurado no `00_Config`.

**Público-alvo:** Engenheiros de Dados e Analistas de Dados

**Agenda:**
| Horário | Duração | Atividade |
| --- | --- | --- |
| 09:00–09:10 | 10 min. | Abertura |
| 09:10–09:25 | 15 min. | Unity Catalog - Governança, Linhagem e Segurança |
| 09:25–10:00 | 35 min. | Lakeflow Designer - ETL Visual (No-code) |
| 10:00–10:20 | 20 min. | Metric View - Camada Semântica Centralizada |
| 10:20–10:40 | 20 min. | SQL Editor |
| 10:40–10:50 | 10 min. | Coffee Break |
| 10:50–11:50 | 60 min. | Genie Agents |
| 11:50–11:55 | 5 min. | AI/BI Dashboard |
| 11:55–12:00 | 5 min. | Genie One |

**Como este roteiro funciona:**
1. Ajustar `catalog` e `schema` no notebook `00_Config`
2. Validar os dados já criados pelo `00_Setup_Workshop`
3. Executar cada etapa do roteiro em sequência
4. Reaproveitar os artefatos criados nas etapas seguintes

**Dados-base já preparados pelo setup:**
| Tabela | Papel no workshop |
| --- | --- |
| `tb_tickets` | Fato principal de tickets de suporte |
| `tb_agents` | Contexto de atendentes e grupos |
| `tb_sla` | Tempos, SLA e satisfação |
| `tb_tickets_comments` | Texto livre para IA e analytics |

> Antes de seguir, execute as células de preparação logo abaixo. Elas validam os dados no schema atual e imprimem prompts já adaptados ao `catalog.schema` configurado.

**Resultado esperado ao final do hands-on:**
* tabelas bronze documentadas no Unity Catalog
* fluxo visual no Lakeflow Designer com saídas Silver/Gold
* uma Metric View reutilizável
* consultas no SQL Editor com AI Functions
* um Genie Agent configurado e testado
* um AI/BI Dashboard criado em cima dos artefatos anteriores

> Fio condutor: começamos com tabelas bronze de suporte de TI, documentamos e governamos no Unity Catalog, transformamos os dados no Designer, padronizamos métricas com Metric View, analisamos no SQL Editor, conversamos com os dados via Genie e finalizamos o consumo em dashboard.

## Preparação do ambiente

### O que cada participante deve fazer agora
1. Abrir o notebook `00_Config`
2. Confirmar ou ajustar `CONFIG_CATALOG` e `CONFIG_SCHEMA`
3. Executar a célula abaixo para carregar a configuração no roteiro
4. Executar a próxima célula para validar as tabelas disponíveis e gerar prompts prontos para copiar

### O que você verá na validação
* contagem das tabelas bronze no schema atual
* nomes totalmente qualificados das tabelas do workshop
* prompts prontos para Unity Catalog, Lakeflow Designer, Metric View, Genie Agents e Dashboard

> A ideia é reduzir retrabalho durante a apresentação: sempre que um passo depender de Genie Code, SQL Editor ou assistente de IA, use os textos gerados automaticamente pela próxima célula.

In [0]:
%run ./00_Config

In [0]:

fq_schema = f"{CATALOG}.{SCHEMA}"

tables = {
    "tb_tickets": f"{fq_schema}.tb_tickets",
    "tb_agents": f"{fq_schema}.tb_agents",
    "tb_sla": f"{fq_schema}.tb_sla",
    "tb_tickets_comments": f"{fq_schema}.tb_tickets_comments",
}

inventory_df = spark.sql(f"""
SELECT tabela, registros
FROM (
    SELECT 'tb_agents' AS tabela, COUNT(*) AS registros FROM {tables['tb_agents']}
    UNION ALL
    SELECT 'tb_sla', COUNT(*) FROM {tables['tb_sla']}
    UNION ALL
    SELECT 'tb_tickets', COUNT(*) FROM {tables['tb_tickets']}
    UNION ALL
    SELECT 'tb_tickets_comments', COUNT(*) FROM {tables['tb_tickets_comments']}
)
ORDER BY tabela
""")

display(inventory_df)

print(f"Schema ativo: {fq_schema}")
print("Tabelas-base do workshop:")
for name, ref in tables.items():
    print(f"- {name}: {ref}")

print("\nAs instruções, prompts e queries de cada etapa estão nas células logo abaixo de cada seção.")
print("Use esta célula apenas para validar o schema ativo e confirmar que os dados base já existem.")


## 1. Unity Catalog — governança aplicada sobre as tabelas bronze

### Entregável da etapa
Ao final desta parte, cada participante deve ter localizado o schema do workshop, inspecionado as tabelas bronze e gerado a descrição da tabela e comentários de colunas usando os recursos nativos do Unity Catalog.

### Passo a passo
1. Abra **Catalog** e navegue até o `catalog.schema` configurado no `00_Config`.
2. Confirme se as tabelas `tb_tickets`, `tb_agents`, `tb_sla` e `tb_tickets_comments` existem.
3. Abra `tb_tickets` ou `tb_agents` e percorra as abas **Overview**, **Columns**, **Sample Data**, **Details**, **History** e **Lineage**.
4. Na própria tela da tabela, use **AI Suggested Description** para sugerir a descrição da tabela.
5. Ainda na mesma tela, use **AI generate** para sugerir comentários das colunas que ainda estão sem comentário.
6. Revise, edite se necessário e aceite as sugestões.
7. Abra a aba **Permissions** e observe como funciona a herança de permissões do catalog/schema e o princípio de menor privilégio.

### Tarefas para os participantes
* Gerar a descrição automática de uma tabela bronze.
* Gerar comentários automáticos para colunas sem descrição.
* Ajustar pelo menos uma sugestão antes de aceitar, para reforçar o papel da curadoria humana.

### Pontos importantes
* A documentação é gerada no próprio Unity Catalog, sem precisar sair para outra ferramenta.
* Os comentários ficam centralizados e reaproveitáveis por SQL, dashboards, Genie e assistentes.
* A aba de lineage ficará mais rica conforme avançarmos nas próximas seções.

## 2. Lakeflow Designer — ETL visual guiado por prompts

### Entregável da etapa
Criar no Lakeflow Designer pelo menos uma tabela Silver e duas tabelas Gold reutilizando o mesmo schema do workshop.

### Passo a passo
1. Acesse **Data Engineering > Visual Data Prep**.
2. **+ NEW**.
3. Abra o assistente do Designer e execute os prompts da célula logo abaixo, na ordem sugerida.
4. A cada prompt, revise o preview antes de confirmar a transformação.
5. Publique ou rode o fluxo e valide se as saídas foram criadas.
6. Volte ao Catalog para conferir a lineage das novas tabelas.

### Detalhe por prompt
* **Prompt 1:** cria a camada Silver com joins e colunas derivadas.
* **Prompt 2:** gera um agregado Gold por país para consumo executivo.
* **Prompt 3:** gera um agregado Gold por atendente/grupo para análise operacional.
* **Prompt 4:** adiciona uma regra simples de qualidade — perceba que o fluxo pode ser refinado sem escrever código.

### Tarefas para os participantes
* Validar se as colunas calculadas de SLA e tempo ficaram corretas.
* Comparar o preview da Silver com o agregado Gold para entender granularidade.
* Publicar o fluxo e conferir se o Designer criou artefatos governados no mesmo schema.

### Validação recomendada
Depois da execução, confira se existem as tabelas `designer_silver_tickets`, `designer_gold_kpis_por_pais` e `designer_gold_performance_atendentes`.

In [0]:
from textwrap import dedent

fq_schema = f"{CATALOG}.{SCHEMA}"
tables = {
    "tb_tickets": f"{fq_schema}.tb_tickets",
    "tb_agents": f"{fq_schema}.tb_agents",
    "tb_sla": f"{fq_schema}.tb_sla",
}

prompts = {
    "designer_prompt_1_silver": dedent(f"""
        Crie uma tabela Silver chamada designer_silver_tickets a partir de {tables['tb_tickets']}.
        Faça join com {tables['tb_agents']} por ticket_id para trazer agent_name, agent_group e agent_interactions.
        Faça join com {tables['tb_sla']} por ticket_id para trazer survey_results, first_response_time, resolution_time,
        expected_sla_to_first_response e expected_sla_to_resolve.
        Crie as colunas tempo_resolucao_horas, tempo_primeira_resposta_horas, dentro_sla_resolucao e dentro_sla_primeira_resposta.
        Padronize priority, status e source com a primeira letra maiúscula.
        Mantenha somente linhas com country não nulo.
    """).strip(),
    "designer_prompt_2_gold_country": dedent(f"""
        A partir de designer_silver_tickets, crie uma tabela Gold chamada designer_gold_kpis_por_pais.
        Agrupe por country e calcule total de tickets, tempo médio de resolução em horas,
        percentual dentro do SLA de resolução e nota média de satisfação.
        Ordene do país com mais tickets para o com menos.
    """).strip(),
    "designer_prompt_3_gold_agent": dedent(f"""
        A partir de designer_silver_tickets, crie uma tabela Gold chamada designer_gold_performance_atendentes.
        Agrupe por agent_group e agent_name e calcule total de tickets, tempo médio de resolução,
        percentual dentro do SLA de resolução, média de interações e nota média de satisfação.
        Ordene pelos atendentes com maior volume de tickets.
    """).strip(),
    "designer_prompt_4_quality": dedent(f"""
        No fluxo atual, adicione uma validação para sinalizar tickets com tempo_resolucao_horas negativo
        ou nulo e exclua esses registros da saída final.
    """).strip(),
}

for title, prompt in prompts.items():
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    print(prompt)


## 3. Metric View — camada semântica reutilizável

### Ideia principal
A Metric View centraliza definições de medidas e dimensões para que SQL, dashboards e Genie usem exatamente a mesma semântica.

### Entregável da etapa
Criar uma Metric View chamada `mv_support_metrics` a partir da tabela `designer_silver_tickets`.

### Passo a passo
1. Use o prompt da célula logo abaixo no Genie Code.
2. Revise se a criação está apontando para o schema correto e se as dimensões e medidas estão em português.
3. Execute a criação da Metric View.
4. No Catalog, abra a Metric View e confirme as seções de **Dimensions** e **Measures**.
5. No SQL Editor, valide com as consultas prontas exibidas na célula logo abaixo, usando `MEASURE()` e `GROUP BY ALL`.

### Tarefas para os participantes
* Confirmar se as medidas e dimensões refletem a linguagem do negócio.
* Ajustar nomes, comentários ou sinônimos se a saída inicial do Genie Code ficar técnica demais.
* Discutir por que uma Metric View reduz retrabalho na criação de dashboards e agentes.

### Observações importantes
* Em consultas, medidas devem ser chamadas com `MEASURE()`.
* `GROUP BY ALL` simplifica o uso das dimensões selecionadas.
* Evite `SELECT *` em Metric Views.

In [0]:
from textwrap import dedent

fq_schema = f"{CATALOG}.{SCHEMA}"

metric_view_prompt = dedent(f"""
    Crie uma Metric View cujo nome DEVE ser exatamente mv_support_metrics no schema {fq_schema}.

    Use como source principal a tabela {fq_schema}.designer_silver_tickets.

    Crie dimensões em português para país, prioridade, status, tópico, grupo de atendimento,
    atendente e mês de abertura.

    Crie medidas para total de tickets, tempo médio de resolução,
    tempo médio de primeira resposta, percentual dentro do SLA de resolução,
    percentual dentro do SLA de primeira resposta, média de interações e nota média de satisfação.

    Adicione comentários, display names e sinônimos úteis para dashboards e Genie.
""").strip()

mv_name = f"{fq_schema}.mv_support_metrics"

validation_prompts = [
    f"Usando a metric view {mv_name}, mostre o total de tickets por país, ordenado do maior para o menor.",
    f"Adicione a query abaixo: Usando a metric view {mv_name}, mostre o percentual de SLA de resolução por prioridade.",
    f"Adicione a query abaixo: Usando a metric view {mv_name}, mostre uma tabela com 3 dimensões (país, prioridade e grupo de atendimento) e 3 medidas (total de tickets, tempo médio de resolução e nota média de satisfação), ordenada pelo total de tickets.",
]

print("=" * 100)
print("PROMPT PARA CRIAÇÃO DA METRIC VIEW (cole no Genie Code)")
print("=" * 100)
print(metric_view_prompt)

print("\n" + "=" * 100)
print("PROMPTS DE VALIDAÇÃO (cole no Genie Code ou SQL Editor após criar a MV)")
print("=" * 100)
for idx, prompt in enumerate(validation_prompts, start=1):
    print(f"\n{idx}. {prompt}")


## 4. SQL Editor — geração assistida e AI Functions

### Entregável da etapa
Executar consultas úteis sobre os dados do workshop e demonstrar pelo menos duas AI Functions aplicadas aos comentários de tickets.

### Passo a passo
1. Abra o **SQL Editor** e selecione o warehouse disponível.
2. Gere uma consulta base com o assistente pedindo um resumo das tabelas do workshop ou usando a Metric View criada na etapa anterior.
3. Em seguida, execute as queries prontas da célula logo abaixo sobre `tb_tickets_comments`.

### Tarefas para os participantes
* Comparar o resultado do assistente com o resultado de uma query escrita manualmente.
* Testar pelo menos uma nova pergunta em linguagem natural para o assistente do SQL Editor.
* Ajustar ou expandir pelo menos uma consulta para responder a uma pergunta nova do grupo.

### Pontos importantes
* O SQL Editor é útil tanto para escrever consultas quanto para refiná-las com contexto do schema.
* AI Functions permitem enriquecer análises textuais sem sair do SQL.

In [0]:
from textwrap import dedent

fq_schema = f"{CATALOG}.{SCHEMA}"

sql_blocks = {
    "6.1 Consultas Básicas": dedent(f"""
        SELECT * FROM {fq_schema}.tb_tickets;
        SELECT * FROM {fq_schema}.tb_agents;
        SELECT * FROM {fq_schema}.tb_sla;
        SELECT * FROM {fq_schema}.tb_tickets_comments;
    """).strip(),
    "6.2 Prompts para o Assistente AI no SQL Editor": dedent(f"""
        Faça a análise de sentimentos da coluna comments da tabela {fq_schema}.tb_tickets_comments
        Faça o join entre {fq_schema}.tb_tickets_comments e {fq_schema}.tb_sla, filtrando survey_results < 3
        Agregue todos os comentários em uma única linha
    """).strip(),
    "6.3 AI Functions - Análise de Sentimento": dedent(f"""
        SELECT
          comments,
          ai_analyze_sentiment(comments) AS sentimento
        FROM {fq_schema}.tb_tickets_comments;
    """).strip(),
    "6.4 AI Functions - Sugestões de Melhoria com ai_query": dedent(f"""
        WITH comentarios_ruins AS (
          SELECT STRING_AGG(tc.comments, ' ') AS all_comments
          FROM {fq_schema}.tb_tickets_comments tc
          JOIN {fq_schema}.tb_sla sc ON tc.ticket_id = sc.ticket_id
          WHERE sc.survey_results < 3
        )
        SELECT
          all_comments,
          ai_query(
            'system.ai.gpt-5-2',
            'Você é um especialista em suporte de TI e precisa trazer 5 sugestões para melhoria de atendimento,
            trazer o peso de importância de cada sugestão, se limite a 1000 caracteres
            e se baseie nos seguintes comentários de atendimentos ruins: ' || all_comments
          ) AS sugestoes
        FROM comentarios_ruins;
    """).strip(),
    "6.5 AI Functions - Sugestões + Nuvem de Palavras": dedent(f"""
        WITH comentarios_ruins AS (
          SELECT STRING_AGG(tc.comments, ' ') AS all_comments
          FROM {fq_schema}.tb_tickets_comments tc
          JOIN {fq_schema}.tb_sla sc ON tc.ticket_id = sc.ticket_id
          WHERE sc.survey_results < 3
        )
        SELECT
          all_comments,
          ai_query(
            'system.ai.gpt-5-2',
            'Você é um especialista em suporte de TI e precisa trazer 5 sugestões para melhoria de atendimento,
            trazer o peso de importância de cada sugestão, se limite a 1000 caracteres
            e se baseie nos seguintes comentários de atendimentos ruins: ' || all_comments
          ) AS sugestoes,
          ai_query(
            'system.ai.gpt-5-2',
            'Você é um especialista em suporte de TI e precisa trazer uma nuvem de palavras
            baseada nos seguintes comentários de atendimentos ruins: ' || all_comments
          ) AS palavras
        FROM comentarios_ruins;
    """).strip(),
}

print("-- 📚 Documentação: https://docs.databricks.com/aws/en/large-language-models/ai-functions")

for title, sql_text in sql_blocks.items():
    print(f"\n-- {'=' * 80}")
    print(f"-- {title}")
    print(f"-- {'=' * 80}")
    print(sql_text)


## 5. Genie Agents — criando a sala manualmente com a Metric View

### Entregável da etapa
Criar uma sala Genie do domínio de suporte usando a Metric View como fonte principal e validá-la com perguntas de negócio.

### Passo a passo
1. No menu lateral, acesse **Genie**.
2. Clique em **New** para criar uma nova sala.
3. Configure o nome e descrião: **IT Support - rcosin**.
4. Em **Data sources**, adicione a Metric View criada na etapa anterior (`mv_support_metrics`).
5. Em **General instructions**, configure:
   * Responder sempre em português.
   * Quando perguntarem sobre tempo de atendimento, formatar em X Dia(s), Y Hora(s), Z Minuto(s).
   * Se perguntarem sobre o pior atendente ou atendente mais lento, responder que não pode fazer tal análise.
6. Salve a sala.

### Perguntas para validar a sala
1. Qual o volume de tickets por país e prioridade?
2. Quais países têm maior percentual fora do SLA?
3. Quais foram os top 10 chamados mais demorados e seus respectivos atendentes?
4. Me adicione também a prioridade, nível de suporte de cada ticket e o grupo de atendimento do atendente.
5. Calcule o percentual de tickets dentro do SLA por país e gere um gráfico de pizza.
6. Qual o pior atendente? *(deve recusar conforme instrução)*

### Pontos importantes
* A sala já funciona com a Metric View sem nenhuma configuração avançada.
* As instruções controlam o comportamento das respostas (idioma, formato, restrições).
* A pergunta 6 valida que a restrição configurada está funcionando.

## 5.1 Melhorando o Genie Agent com o Genie Code

### Ideia principal
O Genie Code é o assistente de IA integrado ao workspace. Dentro de uma sala Genie, ele pode adicionar tabelas, ajustar instruções, criar sample queries e refinar o comportamento do agent — tudo via linguagem natural.

### Entregável da etapa
Usar o Genie Code para enriquecer a sala criada na etapa anterior com mais fontes, instruções detalhadas e sugestões de perguntas.

### Passo a passo
1. Dentro da sala Genie criada na etapa 5, abra o **Genie Code** (painel lateral ou ícone de chat).
2. Cole o prompt da célula logo abaixo.
3. Revise as alterações sugeridas pelo Genie Code antes de aceitar.
4. Salve a sala e teste com as perguntas de validação.

### O que o Genie Code pode fazer na sala
* Adicionar novas tabelas ou Metric Views como fonte.
* Reescrever as instruções gerais com termos de negócio e regras de formato.
* Criar sample queries para guiar os usuários.
* Sugerir perguntas frequentes para exibir na tela inicial.

### Tarefas para os participantes
* Comparar as instruções da sala antes e depois do Genie Code.
* Testar as perguntas de validação e verificar se as respostas melhoraram.
* Ajustar manualmente pelo menos uma instrução para reforçar o papel da curadoria humana.

### Pontos importantes
* O Genie Code entende o contexto da sala e do schema — não é preciso repetir nomes de tabelas.
* A combinação de configuração manual + Genie Code é o fluxo mais produtivo.

In [0]:
fq_schema = f"{CATALOG}.{SCHEMA}"

prompt = f"""Melhore este Genie Agent.
Use como fontes {fq_schema}.mv_support_metrics, {fq_schema}.tb_tickets e {fq_schema}.tb_tickets_comments ou outras tabelas que entender fazerem sentido neste schema.
Configure as instruções para responder sempre em português, usar nomes de métricas amigáveis, filtros, joins, termos de negocio e sugerir perguntas."""

questions = [
    "Qual o volume de tickets por país e prioridade?",
    "Quais países têm maior percentual fora do SLA?",
    "Quais tópicos concentram mais comentários negativos?",
    "Mostre a evolução mensal do total de tickets e do SLA de resolução.",
    "Traga uma análise executiva com principais achados e hipóteses de melhoria.",
]

print("=" * 100)
print("PROMPT PARA O GENIE CODE (cole dentro da sala Genie)")
print("=" * 100)
print(prompt)

print("\n" + "=" * 100)
print("PERGUNTAS SUGERIDAS PARA VALIDAÇÃO")
print("=" * 100)
for idx, q in enumerate(questions, start=1):
    print(f"{idx}. {q}")


## 5.2 Genie como Knowledge Agent — leitura de arquivos de um Volume (Beta)

### Ideia principal
Além de consultar tabelas e Metric Views, o Genie pode ler documentos armazenados em um **Unity Catalog Volume** para responder perguntas sobre políticas, procedimentos e normas internas. Isso transforma a sala em um **Knowledge Agent - RAG** que combina dados estruturados com conhecimento documental.
Necessário:
- Habilitar o Workspace Preview - "Analyze Files in Volumes with Genie Agents"
- Habilitar o Content Search no Volume (se ainda não estiver: configurar em Catalogo > Workspace > "All workspaces have access in" > "This metastore")



### Entregável da etapa
Adicionar um Volume como fonte de dados na sala Genie e validar que ela consegue responder perguntas baseadas nos arquivos.

### Passo a passo
1. Na sala Genie criada anteriormente, clique em **Settings** (engrenagem).
2. Em **Data sources**, clique em **Add data source**.
3. Selecione o tipo **Volume**.
4. Navegue até o volume do workshop no mesmo `catalog.schema` configurado no `00_Config`.
5. Selecione o volume e confirme.
6. Salve as configurações da sala.
7. Aguardar a indexação dos documentos.
8. Teste com as perguntas abaixo.

### Perguntas sobre Políticas e Procedimentos (Knowledge Agent)
1. Quais são as boas práticas de atendimento?
1. Como funciona a escalação entre N1, N2 e N3?

### Análise Integrada: Dados + Políticas (Colaboração Multi-Agente)
1. Quantos tickets P1 estouraram o SLA? E qual deveria ser o prazo de resolução segundo a política?
2. Qual o tempo médio de primeira resposta e o que a política diz sobre isso?
3. Quais países têm mais tickets fora do SLA? E quais são as ações recomendadas pela política?

### Pontos importantes
* O Genie combina dados estruturados (tabelas/MV) com conhecimento não-estruturado (documentos do Volume) na mesma conversa.
* As respostas integradas cruzam métricas reais com normas e políticas internas.
* Funcionalidade em Beta — pode haver limitações de formato ou tamanho de arquivo.

## 6. AI/BI Dashboard — consumo final dos artefatos criados

### Entregável da etapa
Criar um dashboard com indicadores, gráficos e filtros usando preferencialmente a `mv_support_metrics`, e associar a sala Genie criada anteriormente para permitir análise conversacional diretamente no dashboard.

### Passo a passo
1. Acesse **Dashboards** e crie um novo dashboard.
2. Use o prompt da célula logo abaixo.
3. Monte pelo menos:
   * KPIs de volume, SLA, tempo médio de resolução e satisfação
   * visual por país
   * visual por prioridade
   * tabela por atendente ou grupo
4. Adicione filtros de país, prioridade e mês.
5. Revise títulos, formatação e legenda antes de publicar.

### Associando a Sala Genie ao Dashboard
1. No dashboard, clique no ícone de **Settings** (engrenagem) no canto superior.
2. Na aba **General**, localize a seção **Enable Genie** e ative o toggle.
3. Selecione **Link existing Genie agent**.
4. Utilize a URL da sala Genie criada na etapa 5 (IT Support).
5. Salve as configurações.
6. Agora o ícone do Genie aparecerá no dashboard — clique nele e faça uma pergunta para testar.

Isso permite que quem está consumindo o dashboard passe de uma visão estática (gráficos e KPIs) para uma **análise conversacional**, aprofundando qualquer insight diretamente no mesmo contexto de dados.

### Tarefas para os participantes
* Ajustar o tipo visual quando o assistente escolher um gráfico pouco adequado.
* Confirmar se os filtros afetam corretamente os widgets.
* Testar o Genie integrado ao dashboard com uma pergunta de aprofundamento (ex: "quais países têm mais tickets fora do SLA?").
* Se houver tempo, comparar o dashboard com a Metric View e com uma tabela Gold do Designer.

### Reflexão final
Ao concluir o dashboard, observe que ele reutiliza métricas e artefatos criados ao longo do workshop. Com a sala Genie associada, a jornada vai além de visualizações estáticas: qualquer usuário pode continuar a análise de forma conversacional, sem sair do dashboard. Toda a jornada aconteceu na mesma plataforma, de ponta a ponta.

In [0]:
from textwrap import dedent

fq_schema = f"{CATALOG}.{SCHEMA}"

dashboard_prompt = dedent(f"""
    Crie um AI/BI Dashboard chamado IT Support Visão Geral usando {fq_schema}.mv_support_metrics.
    Inclua KPIs de total de tickets, tempo médio de resolução, percentual dentro do SLA e satisfação média.
    Inclua um gráfico por país, um gráfico por prioridade, uma tabela por atendente/grupo e filtros de país, prioridade e mês.
""").strip()

refinements = [
    "Troque o visual de prioridade para pizza",
    "Adicione um ranking de países com pior SLA",
    "Inclua uma tabela apenas com tickets críticos",
    "Formate percentuais com duas casas decimais",
]

print("=" * 100)
print("dashboard_prompt")
print("=" * 100)
print(dashboard_prompt)

print("\nPrompts adicionais de refinamento:")
for idx, refinement in enumerate(refinements, start=1):
    print(f"{idx}. {refinement}")


## 7. Genie One — a porta de entrada para quem só consome

Para usuários que não precisam acessar notebooks, pipelines ou o workspace técnico, o Databricks oferece o **Genie One** — uma interface simplificada onde tudo acontece em um único lugar:

* **Salas Genie** — acesso direto às salas criadas pelo time, sem navegar pelo workspace.
* **Dashboards** — visualização dos AI/BI Dashboards publicados.
* **Agente geral (Genie One)** — um ponto único de conversa que roteia perguntas para as salas e dados corretos automaticamente.
* **Tarefas agendadas** — possibilidade de agendar perguntas recorrentes e receber respostas por e-mail ou notificação.
* **Discovery** — aba de descoberta onde o usuário navega por domínios de dados, encontra tabelas, salas Genie, dashboards e apps disponíveis, sem precisar saber onde cada artefato está no workspace.

O usuário final cai direto nessa experiência, sem precisar entender catalog, schema ou SQL. Tudo o que foi construído ao longo deste workshop — tabelas governadas, Metric Views, salas Genie e dashboards — alimenta essa camada de consumo.